In [ ]:
%%bash
# Colab 2026.07，Linux x86_64。獨立環境，保留筆記本核心。
set -euo pipefail
cd /content
mkdir -p book-tools
curl -fsSL https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C book-tools bin/micromamba
book-tools/bin/micromamba create -y -p /content/book-prodan -c conda-forge python=3.12 psi4 rdkit numpy

## 重繪書圖
作者文章 `site/posts/2024-11-psi4-absorption-spectrum/images/fig03.png` 是計算輸出的曲線圖。內嵌作者藍線中心的像素座標，以 250 與 500 nm 的刻度校正 x 軸；y 軸按最高點正規化。不是實驗圖，也沒有原始躍遷清單。

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# 支援在專案根目錄、book 子目錄或獨立 Colab 工作階段執行。
here = Path.cwd()
book_root = next((p for p in (here, *here.parents) if (p / "outline.yml").is_file()), None)
if book_root is None and (here / "book" / "outline.yml").is_file():
    book_root = here / "book"
fig_dir = (book_root or here) / "figures"
fig_dir.mkdir(exist_ok=True)
# 原圖像素座標，從左往右每隔兩個像素取藍線中心（中位數）。
pixel_x = np.arange(86, 538, 2)
pixel_y = np.array([362.0, 362.0, 362.0, 362.0, 362.0, 361.5, 361.5, 361.5, 361.5, 361.5, 361.5, 361.5, 361.5, 361.5, 361.0, 360.0, 359.0, 356.5, 353.5, 348.5, 342.0, 332.5, 320.5, 305.5, 287.0, 266.5, 243.5, 220.5, 197.5, 175.5, 158.5, 146.5, 141.0, 143.5, 153.0, 168.0, 187.5, 210.0, 233.0, 257.0, 279.0, 298.5, 314.5, 328.0, 338.5, 346.0, 351.0, 355.0, 357.0, 358.5, 359.5, 359.5, 359.5, 358.5, 358.0, 357.0, 356.0, 354.5, 353.0, 350.5, 348.0, 344.5, 341.0, 336.5, 330.0, 323.5, 315.5, 306.0, 295.0, 282.0, 267.0, 251.0, 233.5, 214.0, 193.5, 172.5, 151.0, 130.0, 109.0, 88.5, 70.5, 55.0, 42.5, 33.5, 28.0, 26.0, 29.0, 35.0, 45.0, 58.5, 74.5, 93.0, 113.0, 134.5, 156.0, 178.0, 199.5, 220.5, 240.5, 258.5, 275.0, 290.0, 303.0, 314.0, 323.5, 331.5, 338.0, 343.5, 348.0, 351.5, 354.0, 356.0, 357.5, 359.0, 359.5, 360.5, 360.5, 361.5, 361.5, 361.5, 361.5, 361.5, 361.5, 361.5, 361.5, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0, 362.0])
wavelength_nm = 250 + (pixel_x - 117.5) * (250 / (453 - 117.5))
height = np.maximum(362 - pixel_y, 0)
relative_height = height / height.max()

In [ ]:
fig, ax = plt.subplots(figsize=(7.1, 3.4))
ax.plot(wavelength_nm, relative_height, color="#176d78", linewidth=2.3)
ax.fill_between(wavelength_nm, relative_height, color="#176d78", alpha=0.10)
ax.set(xlabel="Wavelength (nm)", ylabel="Relative height", ylim=(0, 1.09))
ax.set_title("Prodan  |  digitized gas-phase calculation", loc="left")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(fig_dir / "05-05-absorption-spectra.png", dpi=160, bbox_inches="tight")
plt.show()

## 新計算（Colab 2026.07）
執行後會產生本次的獨立結果。原資料及書圖保留在上方。

In [ ]:
%%writefile prodan_live.py
import psi4
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from psi4.driver.procrouting.response.scf_response import tdscf_excitations
from psi4.driver.p4util import spectrum

print("Psi4", psi4.__version__)
rdmol = Chem.AddHs(Chem.MolFromSmiles("CCC(=O)C1=CC2=C(C=C1)C=C(C=C2)N(C)C"))
embedding = AllChem.ETKDGv3()
embedding.randomSeed = 42
if AllChem.EmbedMolecule(rdmol, embedding) != 0:
    raise RuntimeError("無法產生初始構型")
if AllChem.UFFOptimizeMolecule(rdmol, maxIters=1000) != 0:
    raise RuntimeError("初始構型未收斂")
xyz = "\n".join(Chem.MolToXYZBlock(rdmol).splitlines()[2:])
molecule = psi4.geometry("0 1\n" + xyz + "\nunits angstrom\nsymmetry c1")
psi4.set_num_threads(2)
psi4.set_memory("4 GB")
psi4.set_options({"save_jk": True})
psi4.optimize("B3LYP/6-31G(d)", molecule=molecule)
_, wfn = psi4.energy("B3LYP/6-31G(d)", molecule=molecule, return_wfn=True)
states = tdscf_excitations(wfn, states=8, triplets="none")
poles = [state["EXCITATION ENERGY"] for state in states]
moments = [np.linalg.norm(state["ELECTRIC DIPOLE TRANSITION MOMENT (LEN)"])
           for state in states]
curve = spectrum(poles=poles, residues=moments, gamma=0.01, out_units="nm")
np.savetxt("prodan-live-spectrum.csv",
           np.column_stack([curve["convolution"]["x"], curve["convolution"]["y"]]),
           delimiter=",", header="wavelength_nm,epsilon", comments="")
np.savetxt("prodan-live-states.csv",
           np.column_stack([poles, moments, [s["OSCILLATOR STRENGTH (LEN)"] for s in states]]),
           delimiter=",", header="excitation_Eh,moment_au,oscillator_strength", comments="")


In [ ]:
import subprocess
subprocess.run(["/content/book-tools/bin/micromamba", "run", "-p", "/content/book-prodan",
                "python", "prodan_live.py"], check=True)
live = np.loadtxt("prodan-live-spectrum.csv", delimiter=",", skiprows=1)
plt.plot(live[:, 0], live[:, 1])
plt.xlabel("Wavelength (nm)")
plt.ylabel("Molar absorption coefficient (L mol⁻¹ cm⁻¹)")
plt.show()